# Практическая работа 11. ARIMA прогноз кислорода
- Прогнозировать 12 часов
- Сравнить с baseline


## Цель и результат

Прогнозировать 12 часов; Сравнить с baseline.

## Самостоятельное задание

Измените один параметр метода, сравните результат и объясните его значение для аквакультуры.

## Контрольные вопросы

1. Какую ошибку предотвращает использованный протокол?
2. Как интерпретировать результат для специалиста рыбного хозяйства?

## Оценивание

Используйте общую аналитическую рубрику из `assessments/rubric.md`.


In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error,mean_squared_error
df=pd.read_csv("../../data/hydrochem_timeseries.csv",parse_dates=["timestamp"]).set_index("timestamp").asfreq("h")
s=df.dissolved_oxygen_mgL; train,test=s.iloc[:-120],s.iloc[-120:]
preds=[]
for i in range(0,120,12):
 hist=s.iloc[:len(train)+i]; preds.extend(ARIMA(hist,order=(2,1,2)).fit().forecast(12))
rmse=lambda a,b: mean_squared_error(a,b)**.5
print("ARIMA(2,1,2), 12-h blocks: MAE",mean_absolute_error(test,preds),"RMSE",rmse(test,preds))
print("One-step naive (not same horizon): MAE",mean_absolute_error(test,s.shift(1).iloc[-120:]))
# Same-horizon baselines: the forecast is issued every 12 h, like ARIMA
hold=[s.iloc[len(train)+i-1] for i in range(0,120,12) for _ in range(12)]
print("Hold last value for 12 h: MAE",mean_absolute_error(test,hold))
seas=s.shift(24).iloc[-120:]
print("24-h seasonal naive: MAE",mean_absolute_error(test,seas),"RMSE",rmse(test,seas))
